# 05 · Streaming: one decode pass, rally data as soon as the rally ends

`run_stream` emits each rally when the ball has been gone for `rally.end_gap_s` after the last touch, and reports
the latency from the rally's last decoded frame to the moment its events are ready. Identities are provisional
per rally and re-voted at the end (the set-end correction). Use a file to simulate live, or an `srt://` URL.

In [ ]:
import os, sys, json, time
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))
os.chdir(REPO)
print("working in", REPO)

In [ ]:
from vb.config import load_config
from vb.pipeline import run_stream
import numpy as np
cfg = load_config("configs/default.yaml")
cfg.video.source = "s3://YOUR-BUCKET/volleyball/clip.mp4"   # or "srt://host:port" (decoder must be cpu for streams)
cfg.court.seed_pixels, cfg.court.seed_court = [], []        # <- paste from notebook 01
cfg.detector.backend, cfg.pose.backend = "tensorrt", "tensorrt"   # after notebook 03
cfg.output.render = False
def on_rally(r, lat):
    print(f"rally {r.idx:3d} ready {lat:5.1f}s after its last touch: {r.labels}")
res = run_stream(cfg, on_rally=on_rally)
lat = np.array(res.latencies_s)
print(f"median {np.median(lat):.1f}s, p90 {np.percentile(lat, 90):.1f}s over {len(lat)} rallies")